In [ ]:
import os
# Must be set before numpy/scipy/sklearn are imported to prevent
# OpenMP thread conflicts between PyTorch MPS and scikit-learn on macOS ARM
os.environ['OMP_NUM_THREADS']     = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS']     = '1'

import sys
import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from pathlib import Path

project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root / 'src'))
sys.path.insert(0, str(Path.cwd().parent / 'scripts'))

from utilities import KernelNN, setup_device, get_output_affinities

sys.path.append(str(project_root / 'dMRI' / 'scripts'))
from utilities_dMRI import dMRI_Graph_Generator as Data_Generator

data_path = project_root / 'dMRI' / 'data'

device = setup_device()
print(f'Using device: {device}')

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import subprocess


def load_model(model_id: str, model_type: str = 'regular'):
    trained_models_path = project_root / 'dMRI' / 'trained_models' / model_type
    model_path = trained_models_path / f'{model_id}.pth'
    config_path = trained_models_path / 'logs' / 'configs' / f'{model_id}.json'

    with open(config_path, 'r') as f:
        config = json.load(f)

    model = KernelNN(
        width=config['width'],
        ker_width=config['ker_width'],
        depth=config['depth'],
        ker_in=config['edge_attr'],
        in_width=config['node_features'],
        out_width=config['out_width']
    ).to(device)

    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    return model, config

In [ ]:
model, config = load_model(model_id='base_2026-04-29_t11-16-23', model_type='regular')

In [ ]:
N = config["N"]
k = config["k"]
perplexity = config["perplexity"]
sampling_method = config["sampling_method"]
feature_mode = config["feature_mode"]

In [ ]:
subjects = ['115017', '115825', '116726', '118225', '125525']
N_eval = 500

# Set to None to use the whole brain, or pick one of:
# 'frontal', 'parietal', 'temporal', 'occipital', 'subcortical'
# region = 'frontal'
# region = 'parietal'
# region = 'temporal'
# region = 'occipital'
# region = 'subcortical'
region = None


def get_region_mask(normed_coords, region):
    y = normed_coords[:, 1].numpy()
    z = normed_coords[:, 2].numpy()
    masks = {
        'frontal':     y > 0.60,
        'parietal':    (y > 0.35) & (y < 0.85) & (z > 0.62),
        'temporal':    (y > 0.20) & (y < 0.70) & (z < 0.55),
        'occipital':   y < 0.25,
        'subcortical': (y > 0.35) & (y < 0.65) & (z > 0.28) & (z < 0.58),
    }
    if region not in masks:
        raise ValueError(f"Unknown region '{region}'. Choose from: {list(masks.keys())}")
    return masks[region]

In [ ]:
pred_nofe_list = []
pred_pca_list  = []
pred_tsne_list = []
pred_umap_list = []
X_list         = []
edge_idx_list  = []

umap_dir  = str(project_root / 'ERA5' / 'umap')
Generator = Data_Generator(data_path)

try:
    import umap as umap_module
    _umap_direct = True
    print('umap-learn found in current env — running directly')
except ImportError:
    _umap_direct = False
    print('umap-learn not in current env — using conda subprocess fallback')

region_label = region if region else 'whole brain'
print(f'Region: {region_label}\n')

for i, subject in enumerate(subjects):
    print(f'Subject {subject} ({i+1}/{len(subjects)})')

    subject_path = str(data_path / subject / 'T1w' / 'Diffusion_7T') + '/'
    Generator.load_data(subject_path)

    # Optionally restrict to one brain region before sampling
    if region is not None:
        mask = get_region_mask(Generator.normed_coords, region)
        Generator.da           = Generator.da[mask]
        Generator.normed_coords = Generator.normed_coords[mask]
        print(f'  {region}: {mask.sum()} voxels available')

    graph = Generator.sample_graph(
        N=N_eval, k=k, perplexity=perplexity,
        sampling_method=sampling_method, gen_feature_mode=feature_mode
    )

    # NOFE
    print('  NOFE  ...', end='\r')
    pred_nofe = model(graph).detach().cpu().numpy()
    pred_nofe_list.append(pred_nofe)

    # Extract CPU numpy, free GPU
    X        = graph.normed_values.detach().cpu().numpy()
    edge_idx = graph.edge_index.detach().cpu().numpy().T
    X_list.append(X)
    edge_idx_list.append(edge_idx)
    del graph
    if device.type == 'mps':
        torch.mps.synchronize()
        torch.mps.empty_cache()

    # PCA
    print('  PCA   ...', end='\r')
    pred_pca = PCA(n_components=3).fit_transform(X)
    pred_pca_list.append(pred_pca)

    # t-SNE
    print('  t-SNE ...', end='\r')
    pred_tsne = TSNE(n_components=3, perplexity=30, learning_rate='auto',
                     random_state=42).fit_transform(X)
    pred_tsne_list.append(pred_tsne)

    # UMAP
    print('  UMAP  ...', end='\r')
    if _umap_direct:
        reducer = umap_module.UMAP(n_components=3, n_neighbors=15, min_dist=0.1,
                                   metric='euclidean', random_state=42)
        pred_umap = reducer.fit_transform(X)
    else:
        umap_id = f'umap_buffer_{i}'
        np.save(f'{umap_dir}/{umap_id}.npy', X)
        result = subprocess.run(
            f"cd {umap_dir} && conda run -n umap python execute_umap.py {umap_id}",
            shell=True, capture_output=True, text=True
        )
        out_path = f'{umap_dir}/res_{umap_id}.npy'
        if not os.path.exists(out_path):
            print(f'\n  [UMAP subprocess failed]\n  stdout: {result.stdout}\n  stderr: {result.stderr}')
            raise RuntimeError('UMAP subprocess did not produce output file')
        pred_umap = np.load(out_path)
        os.remove(f'{umap_dir}/{umap_id}.npy')
        os.remove(out_path)
    pred_umap_list.append(pred_umap)

    print(f'  Done.')

In [ ]:
i = 0
H = X_list[i]
print(f'H shape: {H.shape}  (voxels x dMRI features)')
print(f'NOFE output shape: {pred_nofe_list[i].shape}')

In [ ]:
def pairwise_distances(X):
    # X: (N, d)
    norms = np.sum(X**2, axis=1, keepdims=True)
    dist2 = norms + norms.T - 2 * X @ X.T
    dist2 = np.maximum(dist2, 0.0)
    return np.sqrt(dist2)

In [ ]:
def stress_1(Y, Z):
    """Stress-1 (global pairwise).
    Y: high-dim input  (N, d)
    Z: low-dim embedding (N, p)
    """
    DY = pairwise_distances(Y)
    DZ = pairwise_distances(Z)
    i  = np.triu_indices_from(DY, k=1)
    dY = DY[i]
    dZ = DZ[i]
    return np.sqrt(np.sum((dY - dZ)**2) / np.sum(dY**2))


def stress_local(Y, Z, edge_indices):
    """Stress-local (graph-edge pairs only).
    edge_indices: (E, 2) array of node index pairs
    """
    dY = np.linalg.norm(Y[edge_indices[:, 0]] - Y[edge_indices[:, 1]], axis=1)
    dZ = np.linalg.norm(Z[edge_indices[:, 0]] - Z[edge_indices[:, 1]], axis=1)
    return np.sqrt(np.sum((dY - dZ)**2) / np.sum(dY**2))


def pairwise_corr(Y, Z):
    DY = pairwise_distances(Y)
    DZ = pairwise_distances(Z)
    i  = np.triu_indices_from(DY, k=1)
    return np.corrcoef(DY[i], DZ[i])[0, 1]

In [ ]:
i = 0
Y = X_list[i]
print(f'Subject {subjects[i]} | NOFE  stress-1={stress_1(Y, pred_nofe_list[i]):.4f}  stress-local={stress_local(Y, pred_nofe_list[i], edge_idx_list[i]):.4f}')

In [ ]:
results = []

method_preds = {
    'NOFE':  pred_nofe_list,
    'PCA':   pred_pca_list,
    't-SNE': pred_tsne_list,
    'UMAP':  pred_umap_list,
}

for i in range(len(subjects)):
    Y        = X_list[i]
    edge_idx = edge_idx_list[i]

    for method, preds in method_preds.items():
        Z = preds[i]
        results.append({
            'subject':      subjects[i],
            'method':       method,
            'pairwise_corr': round(pairwise_corr(Y, Z),   4),
            'stress_1':      round(stress_1(Y, Z),         4),
            'stress_local':  round(stress_local(Y, Z, edge_idx), 4),
        })

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

In [ ]:
summary = df_results.groupby('method')[['pairwise_corr', 'stress_1', 'stress_local']].agg(['mean', 'std'])
print(summary.to_string())

In [ ]:
methods = ['NOFE', 'PCA', 't-SNE', 'UMAP']
colors  = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']

metrics = [
    ('pairwise_corr', 'Pairwise Corr (↑ better)'),
    ('stress_1',      'Stress-1 (↓ better)'),
    ('stress_local',  'Stress-local (↓ better)'),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (metric, title) in zip(axes, metrics):
    vals = [df_results[df_results['method'] == m][metric].values for m in methods]
    means = [v.mean() for v in vals]
    stds  = [v.std()  for v in vals]
    ax.bar(methods, means, color=colors, alpha=0.7)
    ax.errorbar(methods, means, yerr=stds, fmt='none', color='black', capsize=5)
    ax.set_title(title)
    ax.set_ylabel(metric)
    ax.grid(True, alpha=0.3)

plt.suptitle('Distance Preservation — dMRI (HCP)', fontsize=13)
plt.tight_layout()
plt.savefig('distance_preservation_dMRI.png', dpi=150, bbox_inches='tight')
plt.show()